<a href="https://colab.research.google.com/github/cdhaskett/GB885-Final-Project-Haskett-C/blob/main/GB885_Final_Project_Haskett_C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

As a sales analyst for "RUSH," you need to review raw sales data of three tables that include products, retailers and sales. You will need to perform EDA, answer VP questions and identify trends and insights. After that, you need to create a GitHub repository and a recorded presentation. All of these items will be available on the repository.

In [ ]:
#Let's import the needed libraries:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats

In [ ]:
# Load the raw data directly from the GitHub repository
base_url = "https://raw.githubusercontent.com/cdhaskett/GB885-Final-Project-Haskett-C/main/data/"

In [ ]:
# Create separate dataframes for each table
df_products = pd.read_csv(base_url + "TABLE_PRODUCTS_885.csv")
df_retailers = pd.read_csv(base_url + "TABLE_RETAILER_885.csv")
df_sales = pd.read_csv(base_url + "TABLE_SALES_885.csv")

In [ ]:
#Let's take a look at all three dataframe headers:
df_products.head()

In [ ]:
#First thing I noticed is that the products table is pipe delimited. Let's make sure we split them correctly. Since it's all one conglomerate we also need to convert product_id to a numeric type.

In [ ]:
# Split 'PRODUCT_ID|PRODUCT_NAME' into two columns in df_products
df_products[['PRODUCT_ID', 'PRODUCT_NAME']] = df_products['PRODUCT_ID|PRODUCT_NAME'].str.split('|', expand=True)

# Convert PRODUCT_ID to numeric type
df_products['PRODUCT_ID'] = pd.to_numeric(df_products['PRODUCT_ID'])

# Drop the original combined column
df_products = df_products.drop(columns=['PRODUCT_ID|PRODUCT_NAME'])

print("df_products after cleaning:")
display(df_products.head())
print("\nUpdated df_products info:")
df_products.info()

In [ ]:
#Now let's look at df_sales
df_sales.head()
df_sales.info()

In [ ]:
#Right away I see some type mismatches--let's address those.

# Convert 'INVOICE_DATE' to datetime objects in df_sales
df_sales['INVOICE_DATE'] = pd.to_datetime(df_sales['INVOICE_DATE'])

# Convert 'UNITS_SOLD' to numeric type, coercing errors to NaN
df_sales['UNITS_SOLD'] = pd.to_numeric(df_sales['UNITS_SOLD'], errors='coerce')

print("df_sales types after conversion:")
df_sales.info()

In [ ]:
# Check for null values in df_sales
print("Null values in df_sales:")
display(df_sales.isnull().sum())

# Check for duplicate rows in df_sales
print("\nNumber of duplicate rows in df_sales:")
display(df_sales.duplicated().sum())

In [ ]:
#Since we see there are 2 in units_sold and 2 in price_per_unit--let's bring those up. If rows contain valuable information--we will keep them. It looks like there are only a few missing items from those four rows.
missing_values_rows = df_sales[df_sales['PRICE_PER_UNIT'].isnull() | df_sales['UNITS_SOLD'].isnull()]
display(missing_values_rows)

In [ ]:
# Now let's look at the retailer dataframe
df_retailers.head()
df_retailers.info()

In [ ]:
# Check for missing values
df_retailers.isnull().sum()


In [ ]:
# Check for completely duplicated rows
df_retailers.duplicated().sum()

In [ ]:
# Check whether RETAILER_ID is actually unique
df_retailers['RETAILER_ID'].duplicated().sum()

In [ ]:
# Display all records with duplicate retailer IDs
duplicate_retailers = df_retailers[
    df_retailers['RETAILER_ID'].duplicated(keep=False)
].sort_values('RETAILER_ID')

display(duplicate_retailers)

In [ ]:
# Compare retailer ID patterns for the affected retailers
df_retailers[
    df_retailers['RETAILER'].isin(['Sports Direct', 'Walmart', 'West Gear'])
][['RETAILER_ID', 'RETAILER', 'REGION', 'STATE', 'CITY']].sort_values(
    ['RETAILER', 'STATE', 'CITY']
)

The `RETAILER_ID` field was intended to uniquely identify each retailer-location combination. However, the ID appears to be created from abbreviated retailer and location information. In several cases, those abbreviations are not specific enough to remain unique.

For example, Walmart and West Gear in the same city can receive the same `RETAILER_ID`, and the Sports Direct locations in Newark, New Jersey and New York, New York also share an ID.

This indicates that the ID-generation method created collisions in the raw data. Because `RETAILER_ID` is used as the key between the retailer and sales tables, these collisions must be resolved before merging the data.

In [ ]:
# Preserve the original ID and create a cleaned version
df_retailers['RETAILER_ID_CLEAN'] = df_retailers['RETAILER_ID']

In [ ]:
# Create improved IDs only for the duplicate retailer records

state_codes = {
    'Arkansas': 'AR',
    'Florida': 'FL',
    'Texas': 'TX',
    'New Jersey': 'NJ',
    'New York': 'NY'
}

region_codes = {
    'South': 'S',
    'Southeast': 'S',
    'Northeast': 'N'
}

duplicate_mask = df_retailers['RETAILER_ID'].duplicated(keep=False)

df_retailers.loc[duplicate_mask, 'RETAILER_ID_CLEAN'] = (
    df_retailers.loc[duplicate_mask, 'RETAILER']
        .str.replace(' ', '', regex=False)
        .str[:2]
        .str.upper()
    +
    df_retailers.loc[duplicate_mask, 'REGION'].map(region_codes)
    +
    df_retailers.loc[duplicate_mask, 'STATE'].map(state_codes)
    +
    df_retailers.loc[duplicate_mask, 'CITY']
        .str.replace(' ', '', regex=False)
        .str[:3]
        .str.upper()
)

In [ ]:
#Let's check out the changed ids only.
df_retailers.loc[
    duplicate_mask,
    ['RETAILER_ID', 'RETAILER_ID_CLEAN', 'RETAILER', 'STATE', 'CITY']
]

In [ ]:
# Make sure the cleaned IDs are now unique
df_retailers['RETAILER_ID_CLEAN'].duplicated().sum()

To resolve the duplicate retailer IDs, I created a cleaned version of the ID only for the affected records. The revised IDs use additional retailer and location information while preserving the original ID structure as much as possible. After making these corrections, `RETAILER_ID_CLEAN` contained no duplicate values.

In [ ]:
# Preserve the original retailer ID in sales
df_sales['RETAILER_ID_CLEAN'] = df_sales['RETAILER_ID']

### Correcting Retailer IDs in the Sales Table

The duplicate Walmart and West Gear IDs were investigated using the order sequence in the sales table. The affected orders appeared in distinct retailer blocks surrounded by known Walmart or West Gear records.

I used these patterns to assign the affected sales to the appropriate retailer while preserving the original `RETAILER_ID`. The Sports Direct records could not be reliably separated between Newark, New Jersey and New York, New York, so those records were left unresolved rather than assigned arbitrarily.

In [ ]:
# Fix Walmart records
df_sales.loc[
    (df_sales['RETAILER_ID'] == 'W00SFLOR') &
    (df_sales['ORDER_ID'].between(6650, 6975)),
    'RETAILER_ID_CLEAN'
] = 'WASFLORL'

df_sales.loc[
    (df_sales['RETAILER_ID'] == 'W00SARLI') &
    (df_sales['ORDER_ID'].between(6799, 7124)),
    'RETAILER_ID_CLEAN'
] = 'WASARLIT'

df_sales.loc[
    (df_sales['RETAILER_ID'] == 'W00STEHO') &
    (df_sales['ORDER_ID'].between(6909, 7274)),
    'RETAILER_ID_CLEAN'
] = 'WASTXHOU'

In [ ]:
# Fix West Gear records
df_sales.loc[
    (df_sales['RETAILER_ID'] == 'W00SFLOR') &
    (df_sales['ORDER_ID'] >= 7485),
    'RETAILER_ID_CLEAN'
] = 'WESFLORL'

df_sales.loc[
    (df_sales['RETAILER_ID'] == 'W00SARLI') &
    (df_sales['ORDER_ID'] >= 8439),
    'RETAILER_ID_CLEAN'
] = 'WESARLIT'

df_sales.loc[
    (df_sales['RETAILER_ID'] == 'W00STEHO') &
    (df_sales['ORDER_ID'] >= 7473),
    'RETAILER_ID_CLEAN'
] = 'WESTXHOU'

In [ ]:
# Check remaining sales with the original duplicate IDs
df_sales[
    df_sales['RETAILER_ID_CLEAN'].isin(
        ['W00SFLOR', 'W00SARLI', 'W00STEHO', 'S00NNENE']
    )
]['RETAILER_ID_CLEAN'].value_counts()

We have fixed the majority of the issues and are now left with the Sports Direct that are in the New York & New Jersey area.

In [ ]:
#Since we struggled with the cleaning part--let's do the merging in two steps to make sure we can validate each.

# Merge sales with product information
df_merged = df_sales.merge(
    df_products,
    on='PRODUCT_ID',
    how='left',
    validate='m:1'
)

print("Rows after product merge:", len(df_merged))

In [ ]:
# Merge retailer information using the cleaned retailer ID
df_merged = df_merged.merge(
    df_retailers,
    on='RETAILER_ID_CLEAN',
    how='left',
    validate='m:1',
    indicator='retailer_match'
)

print("Rows after retailer merge:", len(df_merged))

In [ ]:
#Let's check the values
df_merged['retailer_match'].value_counts()

In [ ]:
#Let's count the values
df_merged[
    df_merged['retailer_match'] == 'left_only'
]['RETAILER_ID_CLEAN'].value_counts()

In [ ]:
# Investigate the unmatched retailer ID
df_merged[
    df_merged['RETAILER_ID_CLEAN'] == '999999999'
][[
    'ORDER_ID',
    'RETAILER_ID_x',
    'INVOICE_DATE',
    'PRODUCT_ID',
    'PRICE_PER_UNIT',
    'UNITS_SOLD',
    'SALES_METHOD'
]]

One sales record contained the retailer ID `999999999`, which did not match any record in the retailer table. Because the remaining sales information was valid, I kept the transaction for overall sales and product analysis but treated its retailer and location as unknown.

In [ ]:
#Clean up retailer ID columns after the merge
df_merged = df_merged.rename(columns={
    'RETAILER_ID_x': 'RETAILER_ID_ORIGINAL',
    'RETAILER_ID_y': 'RETAILER_ID_RETAILER_TABLE'
})

In [ ]:
#Calculate sales dollars for each order
df_merged['SALES_DOLLARS'] = (
    df_merged['PRICE_PER_UNIT'] *
    df_merged['UNITS_SOLD']
)

In [ ]:
# Check how many orders cannot have sales dollars calculated
df_merged['SALES_DOLLARS'].isnull().sum()

Four sales records could not have total sales dollars calculated because either `PRICE_PER_UNIT` or `UNITS_SOLD` was missing in the raw data. These records were kept in the dataset but will be excluded from calculations that require `SALES_DOLLARS`.

In [ ]:
#Lot's of work--let's check it out.
df_merged[
    ['ORDER_ID', 'PRODUCT_NAME', 'PRICE_PER_UNIT',
     'UNITS_SOLD', 'SALES_DOLLARS']
].head()

In [ ]:
# Final primary key checks
print("Duplicate Order IDs:",
      df_sales['ORDER_ID'].duplicated().sum())

print("Duplicate Product IDs:",
      df_products['PRODUCT_ID'].duplicated().sum())

print("Duplicate Clean Retailer IDs:",
      df_retailers['RETAILER_ID_CLEAN'].duplicated().sum())

In [ ]:
# Confirm the merge did not add or remove sales records
print("Original sales rows:", len(df_sales))
print("Merged rows:", len(df_merged))
print("Unique orders in merged data:", df_merged['ORDER_ID'].nunique())

### Data Cleaning Summary

After reviewing the raw tables, I corrected data type issues, investigated missing values, identified retailer ID collisions, created cleaned retailer IDs for the affected records, and validated the table relationships before merging.

The final merged dataset retained the same number of sales records as the original sales table, confirming that the merge did not duplicate or remove orders. The cleaned data is now ready for sales analysis.

In [ ]:
#Let's first find out what category had teh highest sales in 2021 and how much it sold.

#Filter the data to 2021
df_2021 = df_merged[df_merged['YEAR'] == 2021]

#Calculate sales by product category
product_sales_2021 = (
    df_2021.groupby('PRODUCT_NAME')['SALES_DOLLARS']
    .sum()
    .sort_values(ascending=False)
)

product_sales_2021

In [ ]:
# Display the highest-selling product category
top_product = product_sales_2021.idxmax()
top_product_sales = product_sales_2021.max()

print("Top Product Category:", top_product)
print(f"2021 Sales: ${top_product_sales:,.2f}")

In [ ]:
#What states had teh highest sales of women's products in 2021 and how much was sold?
# Filter to women's products sold in 2021
women_2021 = df_2021[
    df_2021['PRODUCT_NAME'].str.contains("Women's", na=False)
]

# Calculate women's sales by state
women_state_sales = (
    women_2021.groupby('STATE')['SALES_DOLLARS']
    .sum()
    .sort_values(ascending=False)
)

women_state_sales.head(10)

In [ ]:
#Display the top state for women's product sales
top_women_state = women_state_sales.idxmax()
top_women_sales = women_state_sales.max()

print("Top State:", top_women_state)
print(f"Women's Product Sales: ${top_women_sales:,.2f}")

In [ ]:
#What state had the highest sales of men's products in 2021 and how much was sold?

#Filter to men's products sold in 2021
men_2021 = df_2021[
    df_2021['PRODUCT_NAME'].str.contains("Men's", na=False)
]

#Calculate men's sales by state
men_state_sales = (
    men_2021.groupby('STATE')['SALES_DOLLARS']
    .sum()
    .sort_values(ascending=False)
)

men_state_sales.head(10)

In [ ]:
# Display the top state for men's product sales
top_men_state = men_state_sales.idxmax()
top_men_sales = men_state_sales.max()

print("Top State:", top_men_state)
print(f"Men's Product Sales: ${top_men_sales:,.2f}")

In [ ]:
#What retailer prouchased the most units in 2021 and in 2020?

# Calculate units purchased by retailer in 2021
retailer_units_2021 = (
    df_merged[df_merged['YEAR'] == 2021]
    .groupby('RETAILER')['UNITS_SOLD']
    .sum()
    .sort_values(ascending=False)
)

retailer_units_2021

In [ ]:
top_retailer_2021 = retailer_units_2021.idxmax()
top_units_2021 = retailer_units_2021.max()

print("Top Retailer in 2021:", top_retailer_2021)
print(f"Units Purchased: {top_units_2021:,.0f}")

In [ ]:
# Calculate units purchased by retailer in 2020
retailer_units_2020 = (
    df_merged[df_merged['YEAR'] == 2020]
    .groupby('RETAILER')['UNITS_SOLD']
    .sum()
    .sort_values(ascending=False)
)

retailer_units_2020

In [ ]:
top_retailer_2020 = retailer_units_2020.idxmax()
top_units_2020 = retailer_units_2020.max()

print("Top Retailer in 2020:", top_retailer_2020)
print(f"Units Purchased: {top_units_2020:,.0f}")

In [ ]:
#I am a visual person--so let's do some mapping with plotly.

import plotly.express as px

In [ ]:
# Calculate total 2021 sales by state
state_sales_2021 = (
    df_2021.groupby('STATE')['SALES_DOLLARS']
    .sum()
    .reset_index()
)

state_sales_2021.head()

In [ ]:
# Create state abbreviation lookup
state_abbreviations = {
    'Alabama':'AL', 'Alaska':'AK', 'Arizona':'AZ', 'Arkansas':'AR',
    'California':'CA', 'Colorado':'CO', 'Connecticut':'CT', 'Delaware':'DE',
    'Florida':'FL', 'Georgia':'GA', 'Hawaii':'HI', 'Idaho':'ID',
    'Illinois':'IL', 'Indiana':'IN', 'Iowa':'IA', 'Kansas':'KS',
    'Kentucky':'KY', 'Louisiana':'LA', 'Maine':'ME', 'Maryland':'MD',
    'Massachusetts':'MA', 'Michigan':'MI', 'Minnesota':'MN', 'Mississippi':'MS',
    'Missouri':'MO', 'Montana':'MT', 'Nebraska':'NE', 'Nevada':'NV',
    'New Hampshire':'NH', 'New Jersey':'NJ', 'New Mexico':'NM', 'New York':'NY',
    'North Carolina':'NC', 'North Dakota':'ND', 'Ohio':'OH', 'Oklahoma':'OK',
    'Oregon':'OR', 'Pennsylvania':'PA', 'Rhode Island':'RI',
    'South Carolina':'SC', 'South Dakota':'SD', 'Tennessee':'TN',
    'Texas':'TX', 'Utah':'UT', 'Vermont':'VT', 'Virginia':'VA',
    'Washington':'WA', 'West Virginia':'WV', 'Wisconsin':'WI', 'Wyoming':'WY'
}

state_sales_2021['STATE_ABBR'] = (
    state_sales_2021['STATE'].map(state_abbreviations)
)

In [ ]:
# Create a heat map of 2021 sales by state
fig = px.choropleth(
    state_sales_2021,
    locations='STATE_ABBR',
    locationmode='USA-states',
    color='SALES_DOLLARS',
    scope='usa',
    hover_name='STATE',
    hover_data={'SALES_DOLLARS': ':$,.0f', 'STATE_ABBR': False},
    title='2021 Sales Concentration by State'
)

fig.show()

In [ ]:
# Show the top 10 states by 2021 sales
top_states_2021 = (
    state_sales_2021
    .sort_values('SALES_DOLLARS', ascending=False)
    .head(10)
)

top_states_2021[['STATE', 'SALES_DOLLARS']]

In [ ]:
# Create a bar chart of the top 10 states
fig = px.bar(
    top_states_2021,
    x='SALES_DOLLARS',
    y='STATE',
    orientation='h',
    title='Top 10 States by Sales in 2021',
    labels={
        'SALES_DOLLARS': 'Sales Dollars',
        'STATE': 'State'
    }
)

fig.update_layout(
    yaxis={'categoryorder': 'total ascending'}
)

fig.show()

In [ ]:
# Create a clean dataset for the Streamlit dashboard
app_data = df_merged[
    [
        'ORDER_ID',
        'INVOICE_DATE',
        'YEAR',
        'MONTH',
        'PRODUCT_NAME',
        'PRICE_PER_UNIT',
        'UNITS_SOLD',
        'SALES_DOLLARS',
        'OPERATING_MARGIN',
        'SALES_METHOD',
        'RETAILER',
        'REGION',
        'STATE',
        'CITY'
    ]
].copy()

app_data.to_csv('rush_cleaned_sales.csv', index=False)

print("Dashboard dataset created:", len(app_data), "rows")

In [ ]:
from google.colab import files

files.download('rush_cleaned_sales.csv')